# Late deliveries vs. customer satisfaction: charts

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from build_db import build

Path("img").mkdir(exist_ok=True)


def save(fig, name):
    """Show the figure and export it to img/<name>.png (needs kaleido)."""
    fig.show()
    try:
        fig.write_image(f"img/{name}.png", scale=2)
    except Exception as e:  # kaleido missing or no browser available
        print(f"PNG export skipped for {name}: {' '.join(str(e).split())[:140]}")

## Setup: database and query helper

In [2]:
# Load the CSVs from data/ and create the views (same code as `python build_db.py`)
con = build()

# Helper: run a query file from sql/ and return a DataFrame
q = lambda f: con.sql(Path(f"sql/{f}").read_text()).df()

Database ready


## Summary statistics

In [3]:
summary = q("q00_summary.sql")
summary

,n_orders,n_with_review,pct_with_review,pct_late,avg_score
0,96203,95560,99.3,6.79,4.156


## Headline metrics

The numbers quoted in the README (`sql/04_headlines.sql`).

In [4]:
headlines = q("04_headlines.sql")
headlines

,metric_name,metric_value
0,A) % late orders,6.79
1,B) month with highest late rate #1,2018-03 (18.96%)
2,B) month with highest late rate #2,2018-02 (14.13%)
3,B) month with highest late rate #3,2017-11 (12.4%)
4,"C) avg score, 8+ days late",1.7
5,"C) avg score, on time",4.29
6,"D) % of reviews with score 1, 8+ days late",69.81
7,"D) % of reviews with score 1-2, 8+ days late",79.27
8,E) state with highest late rate #1 - AL,21.46
9,E) state with highest late rate #2 - MA,17.53


## 1. How widespread is lateness?

Monthly late rate (top) and number of delivered orders (bottom).

In [5]:
monthly = q("q01_monthly_lateness.sql")
monthly["month"] = pd.to_datetime(monthly["month"])
peak = monthly.loc[monthly["perc_late_orders"].idxmax()]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.65, 0.35], vertical_spacing=0.06)
fig.add_trace(go.Scatter(x=monthly["month"], y=monthly["perc_late_orders"], mode="lines+markers",
                         name="% late"), row=1, col=1)
fig.add_trace(go.Bar(x=monthly["month"], y=monthly["n_orders"], name="Delivered orders"), row=2, col=1)
fig.add_annotation(x=peak["month"], y=peak["perc_late_orders"], showarrow=True, arrowhead=2, ax=-70, ay=-20,
                   text=f"Peak: {peak['perc_late_orders']:.1f}% ({peak['month']:%b %Y})", row=1, col=1)
fig.update_yaxes(title_text="% late orders", rangemode="tozero", row=1, col=1)
fig.update_yaxes(title_text="Delivered orders", row=2, col=1)
fig.update_layout(title="Percentage of late orders and number of delivered orders per month", showlegend=False)
save(fig, "late_rate_monthly")

## 2. Does delay lower the score?

Average score per delay bucket and the full score distribution confirm that delay lowers the score.

In [6]:
by_bucket = q("q02_score_by_delay.sql")

fig = px.bar(by_bucket, x="bucket", y="avg_score", color="bucket",
             text=by_bucket["n"].map("n={:,}".format))
fig.update_yaxes(range=[1, 5], title="Average review score")
fig.update_xaxes(title=None)
fig.update_layout(title="Average score by delay bucket",
                  showlegend=False)
save(fig, "score_by_delay")

In [7]:
dist = q("q03_delay_detail.sql")
dist["score"] = dist["score"].astype(str)

fig = px.bar(dist, x="bucket", y="pct", color="score", barmode="stack",
             category_orders={"score": ["1", "2", "3", "4", "5"]},
             color_discrete_sequence=["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"])
fig.update_yaxes(title="% of reviews")
fig.update_xaxes(title=None)
fig.update_layout(title="Distributions of score for each delay bucket")
save(fig, "score_distribution_by_delay")

## 3. Where? States

Customer state: late rate against average score, bubble size = orders. Dotted lines are the overall values.

In [8]:
states = q("q04_delivery_state.sql")
overall = summary.iloc[0]

fig = px.scatter(states, x="perc_late_orders", y="avg_score", size="n_orders", text="state", hover_name="state",
                 size_max=40)
fig.update_traces(textposition="top center")
fig.add_vline(x=overall["pct_late"], line_dash="dot", line_color="grey")
fig.add_hline(y=overall["avg_score"], line_dash="dot", line_color="grey")
fig.update_xaxes(title="% late orders")
fig.update_yaxes(title="Average review score")
fig.update_layout(title="States: late rate vs average score", showlegend=False)
save(fig, "state_scatter")

## 4. On whom to intervene? Sellers

Grain is (order, seller): an order with items from several sellers counts for each of them. The curves rank sellers by number of late orders: the blue one is their cumulative share of late orders, the orange one is the cumulative share of *all* orders handled by the same sellers. Large sellers have more late orders simply because they ship more, so the gap between the two curves is the real excess concentration of lateness. The table ranks sellers by late rate, among those with at least 30 orders.

In [9]:
sellers = q("q05_sellers_ranking.sql")

ranked = sellers.sort_values("late_orders", ascending=False).reset_index(drop=True)
ranked["seller_share"] = 100 * (ranked.index + 1) / len(ranked)
ranked["cum_late_share"] = 100 * ranked["late_orders"].cumsum() / ranked["late_orders"].sum()
ranked["cum_order_share"] = 100 * ranked["n_orders"].cumsum() / ranked["n_orders"].sum()
k50 = int((ranked["cum_late_share"] >= 50).idxmax()) + 1

fig = go.Figure()
fig.add_trace(go.Scatter(x=ranked["seller_share"], y=ranked["cum_late_share"], mode="lines",
                         name="Cumulative share of late orders"))
fig.add_trace(go.Scatter(x=ranked["seller_share"], y=ranked["cum_order_share"], mode="lines",
                         line=dict(color="orange"), name="Cumulative share of all orders (same sellers)"))
fig.add_trace(go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color="grey", dash="dot"),
                         name="Even distribution"))
fig.update_xaxes(title="% of sellers (ranked by number of late orders)")
fig.update_yaxes(title="% of orders (cumulative)")
fig.update_layout(title=f"{k50} sellers ({100 * k50 / len(ranked):.1f}% of {len(ranked)}) account for half of late orders")
save(fig, "seller_pareto")

# concentration figures to quote in the README: late orders vs all orders of the same sellers
for pct in (10, 20):
    m = ranked["seller_share"] <= pct
    print(f"Top {pct}% of sellers: {ranked.loc[m, 'cum_late_share'].max():.0f}% of late orders, "
          f"{ranked.loc[m, 'cum_order_share'].max():.0f}% of all orders")

top_rate = (sellers[sellers["meets_min_sample"]]
            .sort_values("rank_late_rate")
            .head(10)
            .round({"perc_late_orders": 1}))
top_rate

Top 10% of sellers: 74% of late orders, 63% of all orders
Top 20% of sellers: 87% of late orders, 77% of all orders


,seller_id,n_orders,late_orders,perc_late_orders,meets_min_sample,rank_late_rate
93,ede0c03645598cdfc63ca8237acbe73d,43,14.0,32.6,True,1
115,ad781527c93d00d89a11eecd9dcad7c1,38,12.0,31.6,True,2
49,54965bbe3e4f07ae045b90b0b8541f52,73,22.0,30.1,True,3
95,2a1348e9addc1af5aaa619b1a3679d6b,48,13.0,27.1,True,4
128,835f0f7810c76831d6c7d24c7a646d4d,42,11.0,26.2,True,5
82,beadbee30901a7f61d031b6b686095ad,64,15.0,23.4,True,6
52,a49928bcdf77c55c6d6e05e09a9b4ca5,96,21.0,21.9,True,7
169,ef990a83bbea832f36ebe81376335aa8,43,9.0,20.9,True,8
71,712e6ed8aa4aa1fa65dab41fed5737e4,77,16.0,20.8,True,9
205,71039d19d4303bf9054d69e9a9236699,35,7.0,20.0,True,10


## 5. Is it a category artifact?

Per category: average score for on-time orders vs orders 8+ days late (grain: order and category, so a multi-category order counts in each). Only categories with at least 30 reviewed orders in both groups.

In [10]:
cats = q("q06_category_confounders.sql")
cats["delta"] = cats["avg_score_8_plus_days_late"] - cats["avg_score_on_time"]
cats = cats.sort_values("delta").reset_index(drop=True)   # biggest drop first
n_drop = int((cats["delta"] < 0).sum())

xs, ys = [], []
for _, r in cats.iterrows():
    xs += [r["avg_score_8_plus_days_late"], r["avg_score_on_time"], None]
    ys += [r["category"], r["category"], None]

fig = go.Figure()
fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#cccccc", width=2), showlegend=False))
fig.add_trace(go.Scatter(x=cats["avg_score_on_time"], y=cats["category"], mode="markers",
                         marker=dict(color="grey", size=9), name="On time"))
fig.add_trace(go.Scatter(x=cats["avg_score_8_plus_days_late"], y=cats["category"], mode="markers",
                         marker=dict(color="red", size=9), name="8+ days late"))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title="Average review score", range=[1, 5])
fig.update_layout(title=f"Average score is lower with 8+ days late in {n_drop} of {len(cats)} categories",
                  height=max(400, 24 * len(cats) + 140))
save(fig, "category_dumbbell")

## Caveats

- Association, not causation: late orders also differ in distance, product type and seller.
- Only customers who left a review are in the satisfaction metrics.
- Delivery metrics cover delivered orders only.
- "Late" means delivered on a later calendar day than the estimated delivery date.
- Small groups are excluded (orders < 30)

In [11]:
con.close()